# SSF2 RL — Exploration & Data Collection

This notebook connects to the instrumented SSF2 build, lets you poke at the
observation/action space, drive the character with scripted inputs, and record
trajectories you can later use for supervised learning (behavioral cloning) or
to sanity-check your own RL implementations.

**Prerequisites** (run once, from the repo root):
```bash
cd /Users/cachemiss/Documents/projects/reflash2-fork/reflash2
.venv/bin/pip install -e python          # makes ssf2_rl importable
```

**Before running cells:** start the game in a terminal:
```bash
AIR_SDK_HOME="$HOME/Developer/AIRSDK_51.3.3" bash tools/macos/run_macos.sh
```
The game auto-starts a local VS match and opens the bridge on `127.0.0.1:4567`.

Select the repo's `.venv` as the notebook kernel (Cmd+Shift+P → "Notebook: Select Kernel" → `.venv`).

In [ ]:

import sys
from pathlib import Path

# Make the repo's python package importable regardless of kernel cwd.
REPO = Path("/Users/cachemiss/Documents/projects/reflash2-fork/reflash2")
if str(REPO / "python") not in sys.path:
    sys.path.insert(0, str(REPO / "python"))

import numpy as np
import matplotlib.pyplot as plt

from ssf2_rl.env import SSF2Env, ACTION_NAMES, obs_feature_names

print("actions:", ACTION_NAMES)
print("obs dims:", len(obs_feature_names()))

Obtaining file:///Users/cachemiss/Documents/projects/reflash2-fork/reflash2/python
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ssf2-rl (pyproject.toml) ... done
  Created wheel for ssf2-rl: filename=ssf2_rl-0.1.0-0.editable-py3-none-any.whl size=2792 sha256=31dacefd7a9bc0325deacbdf86d513e6bd8146ef2cf617cebe256c066075d88f
  Stored in directory: /private/var/folders/63/z12mhqls5mx4w2l0sp29qbb40000gn/T/pip-ephem-wheel-cache-x_qvg9ky/wheels/85/26/fb/92b1c9e4622f481c945b421f3c8bf8a7dfa111c5b887579b73
Successfully built ssf2-rl
  Attempting uninstall: ssf2-rl
    Found existing installation: ssf2-rl 0.1.0
    Uninstalling ssf2-rl-0.1.0:
      Successfully uninstalled ssf2-rl-0.1.0
Note: you may need to restart the kernel to use updated packages.
actions: ['noop', 'left', 'right', 'up', 'down', 'jump', 'atta

## 1. Connect & reset

`reset()` restarts the match in-game and takes over player 1, so your inputs
drive Marth while the CPU plays the opponent.

In [ ]:
env = SSF2Env()                      # agent_player=1 by default
obs, info = env.reset()

names = obs_feature_names()
for i, (n, v) in enumerate(zip(names, obs)):
    print(f"{i:2d} {n:16s} {v:+.3f}")
print("\nframe:", info["frame"], "| me:", info["me"]["name"], "| opp:", info["opp"]["name"])

## 2. Scripted control

Run a fixed sequence of named actions and watch the character respond in the
game window. Edit the list to experiment (see `ACTION_NAMES` above).

In [ ]:
# Walk right for 1s, jump-attack, then shield. Each entry = (action_name, frames).
script = [
    ("right", 30),
    ("right_jump", 6),
    ("right_attack", 20),
    ("shield", 15),
    ("noop", 15),
]

xs, ys, dmg = [], [], []
for name, frames in script:
    a = ACTION_NAMES.index(name)
    for _ in range(frames):
        obs, r, term, trunc, info = env.step(a)
        xs.append(info["me"]["x"])
        ys.append(info["me"]["y"])
        dmg.append(info["me"]["damage"])
        if term or trunc:
            obs, info = env.reset()

plt.figure(figsize=(10, 4))
plt.plot(xs, label="me.x")
plt.plot(ys, label="me.y")
plt.legend()
plt.title("Position during scripted control")
plt.xlabel("step")
plt.show()

## 3. Record a random-policy trajectory

Collect `(obs, action, reward, next_obs, done)` tuples with a random policy.
These trajectories are the raw material for behavioral cloning and for
verifying your own value/policy implementations later.

In [ ]:
N_STEPS = 600  # ~20s at 30 FPS

obs, info = env.reset()
traj = {"obs": [], "action": [], "reward": [], "next_obs": [], "done": []}

for t in range(N_STEPS):
    a = env.action_space.sample()
    next_obs, r, term, trunc, info = env.step(a)
    traj["obs"].append(obs)
    traj["action"].append(a)
    traj["reward"].append(r)
    traj["next_obs"].append(next_obs)
    traj["done"].append(term or trunc)
    obs = next_obs
    if term or trunc:
        obs, info = env.reset()

traj = {k: np.asarray(v) for k, v in traj.items()}
print({k: v.shape for k, v in traj.items()})
print("total reward:", round(traj["reward"].sum(), 2), "| mean/step:", round(traj["reward"].mean(), 4))

In [ ]:
# Persist the trajectory for later notebooks / offline analysis.
out = REPO / "notebooks" / "data"
out.mkdir(exist_ok=True)
np.savez_compressed(out / "random_traj.npz", **traj)
print("saved to", out / "random_traj.npz")